# Modul 9: Attention dan Arsitektur Transformer (Capstone)

**Nama:** ISI NAMA  
**NIM:** ISI NIM  
**Kelas:** ISI KELAS  
**Tanggal:** YYYY-MM-DD  

Simpan berkas ini sebagai `M09_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`; jangan menghapus sel pemeriksaan.
2. Gunakan `MODE_TUGAS=False` saat sesi 120 menit. Ubah menjadi `True` untuk
   hasil pengumpulan (6.000/1.500, 4 epoch, 3 seed, dua belas run).
3. Pakai seed individual yang sama dengan Modul 8 agar kedua modul sebanding.
4. Ketiga probe dijalankan pada model **belum terlatih**.
5. Norma gradien dicatat **sebelum** clipping; checkpoint dipilih dari
   validation loss.
6. Luaran: `M09_NIM.ipynb`, `M09_NIM.pdf`, `M09_NIM_metrics.csv`, dan laporan
   maksimal empat halaman.

**Bobot penilaian (total 100).** Pre-lab dan attention dari nol 15; tiga probe
20; anggaran parameter dan perakitan 15; dua belas run multi-seed 25; peta
attention dan biaya panjang 15; keputusan, laporan, dan reproduksibilitas 10.

In [ ]:
import copy
import math
import platform
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

NIM = 'TODO'
BASE_SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
MODE_TUGAS = False  # WAJIB True pada hasil pengumpulan
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_N, VAL_N, EPOCHS = ((6_000, 1_500, 4) if MODE_TUGAS
                          else (2_400, 600, 2))
SEEDS = ([BASE_SEED, BASE_SEED + 1, BASE_SEED + 2]
         if MODE_TUGAS else [BASE_SEED])

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync_device() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

seed_everything(BASE_SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE), 'mode_tugas': MODE_TUGAS,
       'train': TRAIN_N, 'validation': VAL_N, 'epochs': EPOCHS,
       'seeds': SEEDS})

## A. Pre-lab dan attention dari nol - 15 poin

1. **Mengapa skor dot-product dibagi $\sqrt{d_k}$?** TODO
2. **Mengapa tanpa penyandian posisi model tidak dapat membedakan
   "anjing menggigit orang" dari "orang menggigit anjing"?** TODO
3. **Di dua tempat mana mask harus diterapkan?** TODO
4. **Mengapa menambah head tidak menambah parameter bila
   $d_{\text{model}}$ tetap?** TODO

In [ ]:
def attention(Q, K, V, mask=None, skala=True):
    """TODO 1: scaled dot-product attention.

    Q (..., nq, dk); K, V (..., nk, dk); mask True = posisi diabaikan.
    Kembalikan (keluaran, bobot). Terapkan masked_fill SEBELUM softmax.
    """
    raise NotImplementedError

seed_everything(BASE_SEED)
n, d_k = 5, 8
Q, K, V = (torch.randn(1, n, d_k) for _ in range(3))
keluaran, bobot = attention(Q, K, V)
acuan = F.scaled_dot_product_attention(Q, K, V)
selisih = (keluaran - acuan).abs().max().item()
print('bentuk bobot:', tuple(bobot.shape), '| selisih PyTorch:', f'{selisih:.2e}')
assert torch.allclose(bobot.sum(-1), torch.ones(1, n), atol=1e-6), \
    'setiap baris bobot harus berjumlah satu'
assert selisih < 1e-5, 'implementasi belum cocok dengan PyTorch'

In [ ]:
# TODO 2: fungsi entropi_rerata(d_k, skala) yang membangkitkan Q, K, V
#         berukuran (1, 64, d_k), memanggil attention, lalu mengembalikan
#         rerata entropi baris bobot: H = -sum(w * log w).
#         Laporkan tabel untuk d_k in (8, 256) dan skala in (True, False),
#         sertakan kolom entropi maksimum log(64).
raise NotImplementedError

**Interpretasi $\sqrt{d_k}$.** Bandingkan keempat angka entropi terhadap
$\log 64\approx4{,}16$ dan jelaskan akibatnya pada gradien: TODO

## B. Pipeline AG News (disediakan)

Sel berikut menyalin pipeline Modul 7--8 apa adanya agar waktu praktikum
difokuskan pada attention. Jangan mengubah split, sumber vocabulary, atau
indeks token khusus.

In [ ]:
ROOT = Path('../../data/raw/ag_news')
if not ROOT.exists():
    ROOT = Path('data/raw/ag_news')
train_file = ROOT / 'train.csv'
if not train_file.exists():
    raise FileNotFoundError(
        f'{train_file} tidak ditemukan. Ikuti petunjuk data/README.md')

kolom = ['label', 'judul', 'ringkasan']
data = pd.read_csv(train_file, names=kolom, header=None)
if not str(data.iloc[0]['label']).strip().isdigit():
    data = data.iloc[1:].reset_index(drop=True)
data['teks'] = data['judul'].astype(str) + ' ' + data['ringkasan'].astype(str)
data['y'] = data['label'].astype(int) - 1

idx_train, idx_val = train_test_split(
    np.arange(len(data)), train_size=TRAIN_N, test_size=VAL_N,
    stratify=data['y'].to_numpy(), random_state=BASE_SEED)
teks_train = data['teks'].to_numpy()[idx_train]
teks_val = data['teks'].to_numpy()[idx_val]
y_train = data['y'].to_numpy()[idx_train]
y_val = data['y'].to_numpy()[idx_val]

POLA = re.compile(r"[a-z0-9']+")
def tokenisasi(teks):
    return POLA.findall(str(teks).lower())

cacah = Counter(t for s in teks_train for t in tokenisasi(s))
kosakata = ['<pad>', '<unk>'] + [w for w, n_ in cacah.most_common() if n_ >= 2]
stoi = {w: i for i, w in enumerate(kosakata)}
itos = {i: w for w, i in stoi.items()}
V = len(kosakata)

MAKS = 60
def ke_indeks(daftar_teks, maks=MAKS):
    X = torch.zeros(len(daftar_teks), maks, dtype=torch.long)
    L = torch.zeros(len(daftar_teks), dtype=torch.long)
    for i, teks in enumerate(daftar_teks):
        token = [stoi.get(t, 1) for t in tokenisasi(teks)][:maks] or [1]
        X[i, :len(token)] = torch.tensor(token); L[i] = len(token)
    return X, L

X_train, L_train = ke_indeks(teks_train)
X_val, L_val = ke_indeks(teks_val)
ds_train = TensorDataset(X_train, L_train, torch.tensor(y_train))
ds_val = TensorDataset(X_val, L_val, torch.tensor(y_val))
print({'train': len(ds_train), 'validation': len(ds_val), 'vocabulary': V})
assert kosakata[:2] == ['<pad>', '<unk>']
assert L_train.min() >= 1 and L_train.max() <= MAKS

## C. Penyandian posisi dan model - bagian dari 15 poin

In [ ]:
def penyandian_posisi(maks_len, d):
    """TODO 3: penyandian posisi sinusoidal berbentuk (maks_len, d).

    Dimensi genap memakai sin, dimensi ganjil memakai cos, dengan
    frekuensi 1 / 10000^(2i/d). Tidak boleh menambah parameter yang dilatih.
    """
    raise NotImplementedError

PE = penyandian_posisi(256, 100)
assert PE.shape == (256, 100)
assert not torch.allclose(PE[3], PE[7]), 'posisi berbeda harus tersandi berbeda'
assert PE.abs().max() <= 1.0 + 1e-6
print('PE siap:', tuple(PE.shape))

In [ ]:
D_MODEL, N_HEAD, FF, KELAS = 100, 4, 175, 4

class TransformerClassifier(nn.Module):
    def __init__(self, pakai_posisi=True, pakai_mask=True,
                 d=D_MODEL, nhead=N_HEAD, ff=FF, dropout=0.1):
        super().__init__()
        # TODO 4: simpan kedua sakelar; buat Embedding(V, d, padding_idx=0),
        #         nn.TransformerEncoderLayer(batch_first=True), Linear(d, KELAS),
        #         dan daftarkan PE sebagai buffer (bukan parameter).
        raise NotImplementedError

    def masukan(self, X):
        """TODO 5: embedding, ditambah PE bila pakai_posisi aktif."""
        raise NotImplementedError

    def forward(self, X, panjang):
        """TODO 6: jalankan layer encoder lalu pooling.

        Bila pakai_mask: teruskan src_key_padding_mask = (X == 0) dan
        rata-ratakan HANYA token asli (bagi dengan panjang sebenarnya).
        Bila tidak: tanpa mask dan pakai h.mean(1) apa adanya.
        """
        raise NotImplementedError

class LSTMClassifier(nn.Module):
    """Pembanding rekuren Modul 8; disediakan agar protokol identik."""
    def __init__(self, hidden=96, d=D_MODEL):
        super().__init__()
        self.embedding = nn.Embedding(V, d, padding_idx=0)
        self.encoder = nn.LSTM(d, hidden, batch_first=True)
        self.head = nn.Linear(hidden, KELAS)

    def forward(self, X, panjang):
        emb = self.embedding(X)
        packed = pack_padded_sequence(emb, panjang.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, (h_n, _) = self.encoder(packed)
        return self.head(h_n[-1])

seed_everything(BASE_SEED)
m = TransformerClassifier().eval()
with torch.no_grad():
    logits = m(X_val[:4], L_val[:4])
assert logits.shape == (4, KELAS), 'logit harus berbentuk (B, 4)'
assert 'pe' in dict(m.named_buffers()), \
    'PE harus didaftarkan sebagai buffer, bukan parameter yang dilatih'
print('logit:', tuple(logits.shape))

## D. Tiga probe berupa angka - 20 poin

Seluruh probe dijalankan pada model **belum terlatih**: yang diuji arsitektur,
bukan hasil belajar.

In [ ]:
# TODO 7 (probe permutasi): ambil satu contoh terpanjang dari validasi,
#   acak urutan token ASLI saja (bantalan tetap di kanan), lalu hitung
#   selisih maksimum logit untuk pakai_posisi False dan True.
#   Simpan hasilnya pada DataFrame `probe1`.
raise NotImplementedError

print(probe1.to_string(index=False))
assert probe1.loc[0, 'selisih_logit_permutasi'] < 1e-4, \
    'tanpa penyandian posisi seharusnya invarian terhadap permutasi'
assert probe1.loc[1, 'selisih_logit_permutasi'] > 1e-3, \
    'dengan penyandian posisi seharusnya berubah'

In [ ]:
# TODO 8 (probe bantalan): ambil delapan contoh validasi, tambahkan 20 kolom
#   <pad> di kanan, lalu hitung selisih maksimum logit untuk pakai_mask
#   False dan True. Simpan pada DataFrame `probe2`.
raise NotImplementedError

print(probe2.to_string(index=False))
assert probe2.loc[0, 'selisih_logit_bantalan'] > 1e-3, 'tanpa mask harus berubah'
assert probe2.loc[1, 'selisih_logit_bantalan'] < 1e-4, 'dengan mask harus stabil'

In [ ]:
# TODO 9 (probe bobot): panggil layer self_attn secara langsung dengan
#   need_weights=True pada contoh terpanjang, lalu jumlahkan massa bobot yang
#   jatuh pada kolom bantalan, dengan dan tanpa key_padding_mask.
#   Simpan angkanya pada `massa_mask` dan `massa_polos`.
raise NotImplementedError

print(f'massa ke bantalan dengan mask: {massa_mask:.2e} | tanpa mask: {massa_polos:.4f}')
assert massa_mask < 1e-6, 'mask aktif harus membuat massa bobot bantalan nol'

**Bukti angka.** Isi dengan hasil Anda:

- Selisih logit akibat permutasi, tanpa / dengan penyandian posisi: TODO
- Selisih logit akibat penambahan bantalan, tanpa / dengan mask: TODO
- Massa bobot pada posisi bantalan, tanpa / dengan mask: TODO
- Mengapa ketiga kesalahan ini tidak menimbulkan pesan galat? TODO

**Checkpoint menit ke-60.** Tunjukkan kepada asisten: selisih attention
terhadap PyTorch, dua angka entropi, dan ketiga angka probe.

## E. Anggaran parameter - bagian dari 15 poin

In [ ]:
def parameter_layer(d, ff):
    """TODO 10: parameter satu TransformerEncoderLayer.

    Multi-head 4d^2 + 4d; feed-forward 2df + f + d; dua LayerNorm 4d.
    """
    raise NotImplementedError

def parameter_lstm(hidden, d=D_MODEL):
    return 4 * hidden * (d + hidden + 2)

acuan = parameter_lstm(96) + 96 * KELAS + KELAS
head_trf = D_MODEL * KELAS + KELAS

def ff_terdekat(target, d=D_MODEL, batas=1024):
    """TODO 11: integer f yang meminimalkan |parameter_layer(d, f) + head - target|."""
    raise NotImplementedError

FF_CARI = ff_terdekat(acuan)
enc_teori = parameter_layer(D_MODEL, FF_CARI)
seed_everything(BASE_SEED)
enc_nyata = sum(p.numel() for p in
                nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=N_HEAD,
                                           dim_feedforward=FF_CARI,
                                           batch_first=True).parameters())
total_trf = enc_teori + head_trf
print(f'acuan LSTM H=96: {acuan:,} | f terdekat: {FF_CARI} '
      f'| Transformer: {total_trf:,} '
      f'| selisih {100 * (total_trf - acuan) / acuan:+.2f}%')
assert enc_teori == enc_nyata, 'rumus belum cocok dengan PyTorch'
assert FF_CARI == FF, 'f yang ditemukan harus sama dengan FF yang dipakai'
assert abs(total_trf - acuan) / acuan <= 0.01, 'selisih anggaran melebihi 1%'

In [ ]:
# TODO 12 (latihan mandiri): bentuk model untuk n_head in (1, 2, 4, 10)
#   pada d_model tetap, lalu tabelkan d_k dan jumlah parameter layer.
#   Jelaskan mengapa angkanya sama.
raise NotImplementedError

## F. Dua belas run multi-seed - 25 poin

In [ ]:
BATCH = 64

def buat_loader(ds, shuffle, seed, batch=BATCH):
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=generator)

@torch.no_grad()
def evaluasi(model, ds):
    """TODO 13: kembalikan (loss rata-rata, akurasi). Jangan lupa model.eval()."""
    raise NotImplementedError

def bangun(nama, seed):
    seed_everything(seed)
    if nama == 'lstm-h96':
        return LSTMClassifier(96)
    return TransformerClassifier(pakai_posisi=nama != 'trf-tanpa-posisi',
                                 pakai_mask=nama != 'trf-tanpa-mask')

def jalankan(nama, seed):
    """TODO 14: satu fungsi pelatihan untuk SELURUH konfigurasi.

    - Adam lr=1e-3, CrossEntropyLoss, batch 64, clipping 1.0;
    - catat norma gradien SEBELUM clipping;
    - simpan checkpoint dengan validation loss terendah dan muat kembali;
    - ukur waktu dengan sync_device() di kedua ujung;
    - kembalikan (model, history, catatan) dengan catatan memuat run_id, NIM,
      seed, arsitektur, posisi, mask, d_model, n_head, dim_feedforward,
      max_length, ukuran split, epochs, n_updates, encoder_parameters,
      encoder_head_parameters, total_parameters, model_mib_fp32, train_loss,
      val_loss, val_accuracy, grad_norm_mean, seconds_per_epoch, device, notes.
    """
    raise NotImplementedError

KONFIG = ['lstm-h96', 'trf-tanpa-posisi', 'trf-tanpa-mask', 'trf-penuh']
hasil, riwayat, simpan = [], {}, {}
for seed in SEEDS:
    for nama in KONFIG:
        print(f'Melatih {nama}, seed={seed} ...')
        mdl, hist, row = jalankan(nama, seed)
        hasil.append(row); riwayat[(nama, seed)] = hist; simpan[nama] = mdl

tabel = pd.DataFrame(hasil)
print(tabel[['run_id', 'encoder_head_parameters', 'total_parameters',
             'val_loss', 'val_accuracy', 'grad_norm_mean',
             'seconds_per_epoch']].to_string(index=False))
assert len(tabel) == len(KONFIG) * len(SEEDS)
assert tabel.groupby('seed')['n_updates'].nunique().max() == 1, \
    'seluruh konfigurasi harus memakai anggaran update yang sama'
if MODE_TUGAS:
    assert len(tabel) == 12, 'hasil pengumpulan wajib memuat dua belas run'

In [ ]:
# TODO 15: dua panel — validation accuracy dan validation loss per epoch
#   untuk keempat konfigurasi pada BASE_SEED. Beri label sumbu, legenda, grid.
raise NotImplementedError

In [ ]:
# TODO 16: ringkasan per konfigurasi berisi acc_mean, acc_std (ddof=0),
#   seconds_mean, encoder_head_parameters, total_parameters, dan model_mib.
#   Buat bar chart acc_mean dengan error bar satu simpangan baku dan
#   anotasi waktu per epoch. Simpan tabelnya sebagai `ringkasan`.
raise NotImplementedError

## G. Peta attention dan biaya panjang - 15 poin

In [ ]:
# TODO 17: ambil satu contoh validasi dengan minimal 12 token, hitung bobot
#   attention model `trf-penuh` terlatih (need_weights=True, dengan mask),
#   buktikan massa bobot ke kolom bantalan bernilai nol, lalu tampilkan peta
#   n x n dengan label token pada kedua sumbu.
raise NotImplementedError

**Dua pasangan token berbobot tertinggi.** Sebutkan pasangannya dan nilai
apakah keduanya masuk akal secara linguistik. Ingat: bobot attention adalah
petunjuk aliran informasi, bukan penjelasan sebab-akibat. TODO

In [ ]:
# TODO 18: ukur waktu satu forward pass (batch 64) untuk n in (30, 60, 120)
#   pada LSTM dan Transformer terlatih. Lakukan satu pemanggilan pemanasan,
#   ulangi minimal 20 kali, dan sinkronkan device. Tabelkan waktu beserta
#   rasionya terhadap n = 30, lalu simpan sebagai `biaya`.
raise NotImplementedError

In [ ]:
# TODO 19: simpan seluruh run ke metrics.csv.
if MODE_TUGAS:
    tabel.insert(1, 'student_id', NIM)
    output = Path(f'M09_{NIM}_metrics.csv')
    tabel.to_csv(output, index=False)
    print(f'{len(tabel)} baris disimpan ke {output}')
else:
    print('MODE_TUGAS masih False; metrics.csv belum ditulis.')

## H. Pertanyaan analisis dan keputusan - 10 poin

1. Berapa selisih akurasi Transformer penuh terhadap LSTM, dan apakah lebih
   besar daripada variasi antar-seed? **TODO**
2. Berapa penurunan akurasi akibat mematikan penyandian posisi, dan bagaimana
   kaitannya dengan probe permutasi? **TODO**
3. Berapa penurunan akurasi akibat mematikan mask, dan mengapa kesalahan ini
   tidak menimbulkan pesan galat? **TODO**
4. Arsitektur mana yang lebih cepat pada $n=60$, dan bagaimana urutannya pada
   $n=120$? Kaitkan dengan $O(n)$ lawan $O(n^2)$ memakai angka sendiri. **TODO**
5. Jika model harus melayani teks jauh lebih panjang dengan batas memori tetap,
   arsitektur mana yang Anda pilih? Gunakan sedikitnya tiga angka. **TODO**

## Checklist sebelum mengumpulkan

- [ ] Identitas terisi dan `MODE_TUGAS=True`.
- [ ] Attention buatan sendiri cocok dengan PyTorch di bawah $10^{-5}$.
- [ ] Empat angka entropi dilaporkan terhadap $\log 64$.
- [ ] Ketiga probe dijalankan pada model belum terlatih dan berupa angka.
- [ ] Selisih anggaran encoder + head terhadap LSTM tidak lebih dari 1%.
- [ ] Dua belas run tercatat, termasuk kedua ablasi yang benar-benar dilatih.
- [ ] Norma gradien dicatat sebelum clipping.
- [ ] Peta attention memperlihatkan kolom bantalan bernilai nol.
- [ ] Waktu forward pass diukur untuk $n \in \{30, 60, 120\}$.
- [ ] Test set tidak dipakai untuk tuning atau pemilihan model.
- [ ] `metrics.csv` memuat seluruh run.
- [ ] Notebook lolos *Restart Kernel and Run All*.